# Tableau Dashboard Data Preparation

Prepare a dashboard-ready Spotify dataset for Tableau using the same
track-level deduplication method used in the analysis notebooks.

## Input
- spotifydataset_Visualisation.csv

## Output
- spotify_dashboard_data.csv

In [1]:
import pandas as pd
from pathlib import Path

# project root is one folder above jupyter_notebooks
project_root = Path.cwd().parent

## Load Dataset

In [2]:
# find the visualisation dataset in the project
csv_path = next(project_root.rglob("spotifydataset_Visualisation.csv"))

# read csv into DataFrame
dfSpotify = pd.read_csv(csv_path)

print(f"File: {csv_path}")
print(f"Starting shape: {dfSpotify.shape}")

dfSpotify.head()

File: /Users/sahraosman/Documents/vscode-projects/Spotify Music Trend Analysis/assets/csv/VisualisationFiles/spotifydataset_Visualisation.csv
Starting shape: (113999, 21)


,Unnamed: 0.1,Unnamed: 0,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Remove Duplicate Track Records

In [3]:
# sort by popularity and remove duplicate tracks
dfDashboard = (
    dfSpotify.sort_values(by="popularity", ascending=False)
    .drop_duplicates(subset=["artists", "album_name", "track_name"], keep="first")
    .copy()
)

# remove unnecessary index columns
dfDashboard = dfDashboard.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])

print(f"Starting rows: {len(dfSpotify)}")
print(f"Dashboard rows: {len(dfDashboard)}")
print(f"Rows removed: {len(dfSpotify) - len(dfDashboard)}")

dfDashboard.head()

Starting rows: 113999
Dashboard rows: 89379
Rows removed: 24620


,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
20001,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),Unholy (feat. Kim Petras),100,156943,False,0.714,0.472,2,-7.375,1,0.0864,0.01300,0.000005,0.266,0.238,131.121,4,dance
51664,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52","Quevedo: Bzrp Music Sessions, Vol. 52",99,198937,False,0.621,0.782,2,-5.548,1,0.0440,0.01250,0.033000,0.230,0.550,128.033,4,hip-hop
89410,Manuel Turizo,La Bachata,La Bachata,98,162637,False,0.835,0.679,7,-5.329,0,0.0364,0.58300,0.000002,0.218,0.850,124.980,4,reggaeton
81209,David Guetta;Bebe Rexha,I'm Good (Blue),I'm Good (Blue),98,175238,True,0.561,0.965,7,-3.673,0,0.0343,0.00383,0.000007,0.371,0.304,128.040,4,pop
68303,Bad Bunny,Un Verano Sin Ti,Tití Me Preguntó,97,243716,False,0.650,0.715,5,-5.198,0,0.2530,0.09930,0.000291,0.126,0.187,106.672,4,latino


## Validate Dashboard Dataset

In [4]:
# check remaining duplicates
remaining_duplicates = dfDashboard.duplicated(
    subset=["artists", "album_name", "track_name"]
).sum()

# check missing values
missing_values = dfDashboard.isnull().sum().sum()

print(f"Remaining duplicate tracks: {remaining_duplicates}")
print(f"Missing values: {missing_values}")
print(f"Final shape: {dfDashboard.shape}")

Remaining duplicate tracks: 0
Missing values: 0
Final shape: (89379, 19)


In [5]:
# check average popularity by explicit status
dfDashboard.groupby("explicit")["popularity"].mean()

explicit
False    32.878003
True     36.942006
Name: popularity, dtype: float64

In [6]:
# check top genres by average popularity
(
    dfDashboard.groupby("track_genre")["popularity"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

track_genre
k-pop                58.906250
pop-film             57.418972
chill                53.676884
sad                  51.912258
singer-songwriter    50.732899
grunge               50.261029
hard-rock            50.218695
progressive-house    49.708738
indian               49.594828
metal                49.314103
Name: popularity, dtype: float64

## Export Dashboard Dataset

In [7]:
# save dashboard-ready dataset in the visualisation data folder
output_path = csv_path.parent / "spotify_dashboard_data.csv"

dfDashboard.to_csv(output_path, index=False)

print(f"Saved successfully:\n{output_path}")

Saved successfully:
/Users/sahraosman/Documents/vscode-projects/Spotify Music Trend Analysis/assets/csv/VisualisationFiles/spotify_dashboard_data.csv


---

# Conclusion

The Spotify visualisation dataset was prepared for use in the Tableau dashboard.

Duplicate track records were removed using artist, album name and track name,
with the highest-popularity record retained where duplicates existed.

The final dataset contains 89,379 unique track records with no missing values or remaining
duplicate track records.

Validation checks confirmed that the results match the analysis notebooks.

The dashboard-ready dataset was exported as:

`spotify_dashboard_data.csv`